In [48]:
import pandas as pd
import numpy as np
from pathlib import Path

1. Load Dataset

In [49]:
print("Loading Dataset...")
BASE_DIR = Path.cwd().resolve().parent
DATASET_PATH = BASE_DIR / "docs" / "dataset" / "Ethereum_transactions.csv"
df = pd.read_csv(DATASET_PATH)
print(f"Dataset Loaded Successfully! Initial Shape: {df.shape}\n")

Loading Dataset...
Dataset Loaded Successfully! Initial Shape: (1184693, 33)



2. Initial Dataset Inspection

In [50]:
print("Initial Inspection of Dataset:")
print(df.info())
print("\nFirst 5 Rows of the Dataset:")
df.head(5)

Initial Inspection of Dataset:
<class 'pandas.DataFrame'>
RangeIndex: 1184693 entries, 0 to 1184692
Data columns (total 33 columns):
 #   Column                        Non-Null Count    Dtype  
---  ------                        --------------    -----  
 0   hash                          1184693 non-null  str    
 1   nonce                         1184693 non-null  int64  
 2   transaction_index             1184693 non-null  int64  
 3   from_address                  1184693 non-null  str    
 4   to_address                    1180660 non-null  str    
 5   value                         1184693 non-null  float64
 6   gas                           1184693 non-null  int64  
 7   gas_price                     1184693 non-null  int64  
 8   receipt_gas_used              1184693 non-null  int64  
 9   receipt_cumulative_gas_used   1184693 non-null  int64  
 10  block_timestamp               1184693 non-null  str    
 11  block_number                  1184693 non-null  int64  
 12  receipt_

,hash,nonce,transaction_index,from_address,to_address,value,gas,gas_price,receipt_gas_used,receipt_cumulative_gas_used,...,avg_time_between_sent_tx,sent_tx_count_so_far,avg_value_sent_so_far,std_value_sent_so_far,value_zscore,gas_price_deviation,failed_tx_ratio,avg_time_between_received_tx,received_tx_count_so_far,avg_value_received_so_far
0,0xfce3d48a4d3a20fe0ee3aba49e8fd8ec862475e22871...,131517,9,0xdd07249e403979bd79848c27aa5454c7e66bdee7,0x000000000000084e91743124a982076c59f10084,0.0,40000,302428763434,14627,1195743,...,5.617000e+03,172,0.0,0.0,NaN,1.959482e+11,0.0,4104.730769,235.0,0.0
1,0x828a041bdc4172384bc234d6300cfa6fad54f5add9de...,41547,113,0x9c89814bc229ef3188add441ef9d2673a621d7be,0x000000109099b65ba98f0934ddd9c95e8ea3fd48,0.0,380000,5573099902,16424,5939065,...,1.841612e+06,18,0.0,0.0,NaN,-2.533746e+10,0.0,14212.714286,8.0,0.0
2,0x05d4cca71986157bbd40aafc973614a6593f8283ddbe...,131206,9,0xdd07249e403979bd79848c27aa5454c7e66bdee7,0x000000000000084e91743124a982076c59f10084,0.0,176934,335029126401,16579,753251,...,5.918033e+03,154,0.0,0.0,NaN,2.265093e+11,0.0,4271.674641,210.0,0.0
3,0x64655a49040787ba2e5e73b19b02b32f4b672e9cf836...,34125,82,0xff6d62bc882c2fca5af5cbfe1e6c10b97ba251a4,0x62ce6843abea82e09652567be3b41a271678b403,0.0,820000,10000000000,17294,4350946,...,7.434080e+03,76,0.0,0.0,NaN,3.723684e+09,0.0,328.220657,214.0,0.0
4,0xe78b6431ebce4f2e6aa20938c9efc07485e98a68d035...,110743,25,0x05be6e1f661dacd4630e1ebe2ffce5bfb962076f,0xccd5e8d817e3fb420dd0da9cbbb1cd94d9934417,0.0,360000,46474484767,18223,1177062,...,3.991746e+03,174,0.0,0.0,NaN,1.549152e+10,0.0,2970.454545,12.0,0.0


3. Duplicate Analysis

In [51]:
print("\nChecking for Duplicate Rows:")
duplicate_rows = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_rows)


Checking for Duplicate Rows:
Number of duplicate rows: 29370


4. Duplicate Removal

In [52]:
print("Removing duplicate rows...")
if duplicate_rows > 0:
    df.drop_duplicates(inplace=True)
    print("Number of rows after removing duplicates:", len(df))
else:
    print("No duplicate rows found.")

Removing duplicate rows...
Number of rows after removing duplicates: 1155323


5. Missing Value Analysis

In [53]:
print("Missing Values Check:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

Missing Values Check:
to_address                         4033
receipt_contract_address        1151290
max_fee_per_gas                  772649
max_priority_fee_per_gas         772649
to_address_age_seconds             4033
receiver_is_newly_created          4033
time_since_last_sent_tx          674539
avg_time_between_sent_tx         784691
avg_value_sent_so_far            674539
std_value_sent_so_far            784691
value_zscore                     985993
gas_price_deviation              674539
failed_tx_ratio                  674539
avg_time_between_received_tx     537611
received_tx_count_so_far           4033
avg_value_received_so_far        478847
dtype: int64


6. Missing value Treatment

In [54]:
print("\n Handling Missing Values:")
# 1. Address age column missing
age_cols=['from_address_age_seconds','to_address_age_seconds']
# Unknown history indicator
for col in age_cols:
    if col in df.columns:
        df[col]=df[col].fillna(-1)
# Gas fee columns missing
gas_cols=['max_fee_per_gas','max_priority_fee_per_gas']
for col in gas_cols:
    if col in df.columns:
        df[col]=df[col].fillna(df[col].median())
# First transaction indicators
df['first_sent_tx']=df['time_since_last_sent_tx'].isnull().astype(int)
df['first_received_tx']=df['avg_time_between_received_tx'].isnull().astype(int)
# Fill missing time-related columns with -1 to indicate unknown
time_cols=['time_since_last_sent_tx','avg_time_between_sent_tx','avg_time_between_received_tx']
df[time_cols]=df[time_cols].fillna(-1)
# Fill missing avg_value_received_so_far with 0
df['avg_value_received_so_far']=df['avg_value_received_so_far'].fillna(0)
print("Missing values treatment complete.\n")


 Handling Missing Values:
Missing values treatment complete.



7. Contract Creation Handling

In [55]:
print("\n Handling contract creation cases:")
# mask for contract creation transactions
contract_creation_mask = df['to_address'].isnull() & (df['is_contract_call']==0)
# Case 1: Smart Contract creation rows
df.loc[contract_creation_mask, 'to_address_age_seconds'] = 0
df.loc[contract_creation_mask, 'receiver_is_newly_created'] = 1
df.loc[contract_creation_mask, 'to_address'] = '0x0000000000000000000000000000000000000000'
df.loc[contract_creation_mask & df['receipt_contract_address'].isnull(), 'receipt_contract_address'] = 'Created Contract Address Missing'
# Case 2: Non contract creation rows with null to_address
df.loc[~contract_creation_mask & df['to_address'].isnull(), 'receiver_is_newly_created'] = 0
df.loc[~contract_creation_mask & df['to_address'].isnull(), 'to_address'] = 'Unknown Receiver'
df.loc[~contract_creation_mask & df['receipt_contract_address'].isnull(), 'receipt_contract_address'] = 'No Contract Created'
print("Contract creation handling complete.\n")


 Handling contract creation cases:
Contract creation handling complete.



8. Unknown / Invalid values handling

In [66]:
print("\nHandling Unknown/Invalid Values:")
# Receiver transaction history count is missing / unknown
if 'received_tx_count_so_far' in df.columns:
    df['received_tx_count_so_far'] = df['received_tx_count_so_far'].fillna(0).astype(int)

# Sender missing / unknown
if 'from_address' in df.columns:
    df['from_address'] = df['from_address'].fillna('Unknown Sender')
print("Unknown and invalid value handling complete.\n")
# Specific statistical & ratio features cleanup
zero_fill_cols = [
    'avg_value_sent_so_far', 
    'std_value_sent_so_far', 
    'gas_price_deviation', 
    'failed_tx_ratio'
]

for col in zero_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)
# Handle NaN values in z-score columns
zscore_cols = [col for col in df.columns if 'zscore' in col]

for col in zscore_cols:
    df[col] = df[col].fillna(0)



Handling Unknown/Invalid Values:
Unknown and invalid value handling complete.



9. Data Validation Checks

In [68]:
# Data validation checks
print('\n Running Data Validation Checks...')
# 1. Total missing values check
total_missing_values = df.isnull().sum().sum()
print(f"Total missing values in the dataset: {total_missing_values}")
# 2. Duplicate rows check
total_duplicates=df.duplicated().sum()
print(f"Number of duplicate rows remaining: {total_duplicates}")
# 3. Binary columns validation
binary_columns = ['is_contract_call', 'from_is_contract', 'to_is_contract', 'receiver_is_newly_created', 'is_contract_created']
print("\nBinary Columns Validation:")
for col in binary_columns:
    if col in df.columns:
        unique_values = df[col].unique()
        if set(unique_values).issubset({0, 1}):
            print(f"Column '{col}' is valid with unique values: {unique_values}")
        else:
            print(f"Column '{col}' has invalid values: {unique_values}")
    else:
        print(f"Column '{col}' not found in the dataset.")
# 4. Numeric columns validation
numeric_checks = {
    'value': (df['value'] < 0).sum() if 'value' in df.columns else 0,
    'gas': (df['gas'] < 0).sum() if 'gas' in df.columns else 0,
    'gas_price': (df['gas_price'] < 0).sum() if 'gas_price' in df.columns else 0,
    'receipt_gas_used': (df['receipt_gas_used'] < 0).sum() if 'receipt_gas_used' in df.columns else 0,
    'nonce': (df['nonce'] < 0).sum() if 'nonce' in df.columns else 0
}
print("\nNumeric Columns Validation:")
for col, count in numeric_checks.items():
    if count > 0:
        print(f"Column '{col}' has {count} negative values.")
# 5. Logical relationship checks
print("\nLogical Relationship Checks:")
if 'receipt_gas_used' in df.columns and 'gas' in df.columns:
    invalid_gas_used = (df['receipt_gas_used'] > df['gas']).sum()
    print(f"Number of rows where receipt_gas_used > gas: {invalid_gas_used}")
# 6. Age flags validation
if 'from_address_age_seconds' in df.columns and 'to_address_age_seconds' in df.columns:
    invalid_from_age = (df['from_address_age_seconds'] < -1).sum() + df['from_address_age_seconds'].isnull().sum()
    invalid_to_age = (df['to_address_age_seconds'] < -1).sum() + df['to_address_age_seconds'].isnull().sum()
    print(f"Number of rows with invalid from_address_age_seconds: {invalid_from_age}")
    print(f"Number of rows with invalid to_address_age_seconds: {invalid_to_age}")

print("Data validation checks completed.\n")


 Running Data Validation Checks...
Total missing values in the dataset: 0
Number of duplicate rows remaining: 0

Binary Columns Validation:
Column 'is_contract_call' is valid with unique values: [1 0]
Column 'from_is_contract' is valid with unique values: [0]
Column 'to_is_contract' is valid with unique values: [1 0]
Column 'receiver_is_newly_created' is valid with unique values: [0. 1.]
Column 'is_contract_created' is valid with unique values: [1]

Numeric Columns Validation:

Logical Relationship Checks:
Number of rows where receipt_gas_used > gas: 0
Number of rows with invalid from_address_age_seconds: 0
Number of rows with invalid to_address_age_seconds: 0
Data validation checks completed.



10. Final Dataset Vaidation

In [79]:
print("="*50)
print("     ----- Final Dataset Summary -----   ")
print("="*50)
# 1. Total Missing Values Check Across Entire DataFrame
total_nulls = df.isnull().sum().sum()
print(f"1. Total Remaining Null/Missing Values: {total_nulls}")
# 2. Duplicate Rows Check
duplicate_rows = df.duplicated().sum()
print(f"2. Total Duplicate Rows: {duplicate_rows}")
# 3. Data Types Integrity Check
print("3. Column Data Types Summary:")
print(df.dtypes.value_counts())
# Final shape aur summary verify karein
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print("Dataset fully cleaned, validated, and ready for export!")

     ----- Final Dataset Summary -----   
1. Total Remaining Null/Missing Values: 0
2. Total Duplicate Rows: 0
3. Column Data Types Summary:
int64      18
float64    14
str         5
Name: count, dtype: int64
Total Rows: 1155323
Total Columns: 37
Dataset fully cleaned, validated, and ready for export!


11. Exporting Cleaned data

In [81]:
# EXPORT CLEANED DATASET AS CSV
import os
print("="*50)
print("     ----- EXPORT CLEANED DATASET -----   ")
print("="*50)
project_root = os.path.dirname(os.getcwd())
output_dir = os.path.join(project_root, "docs", "dataset")
output_filename = "cleaned_blockchain_tx_dataset.csv"
full_path = os.path.join(output_dir, output_filename)
df.to_csv(full_path, index=False)
print(f"Cleaned dataset successfully exported to: {output_filename}")
print(f"Final Data Shape: {df.shape[0]} rows, {df.shape[1]} columns")

     ----- EXPORT CLEANED DATASET -----   
Cleaned dataset successfully exported to: cleaned_blockchain_tx_dataset.csv
Final Data Shape: 1155323 rows, 37 columns
